# REAL PTT Cloud Platform - Complete Download Tool v4

Downloads **all three data types** from the REAL PTT platform:

| Data Type | Source | Endpoint | Format |
| --- | --- | --- | --- |
| **Video** (PTT video calls) | Web HTML | `realptt.com/ptt/webserver?event=org_videolist` | Parse `Ud()` JS calls → direct MP4 URLs |
| **Audio** (PTT radio recordings) | Web HTML | `realptt.com/ptt/webserver?event=org_audiolist` | Parse `Ud()` JS calls → construct WAV URLs |
| **Upload Files** (body cameras, pictures) | JSON API | `realptt.com/ptt/uploadFile?method=get` | JSON response (limit=20 per page) |

### Key Technical Details
- **Domain**: `realptt.com` (NOT `api.realptt.com` — the API domain returns incomplete data)
- **Video**: Embedded in HTML as `Ud(id, time, userId, ..., url, ...)` JS calls
- **Audio**: Requires per-group per-day queries. Download URL: `record.realptt.com/voice/?SpkId=...`
- **Audio time conversion**: Server time = Client time − 13 hours (timeDifference=46800000ms for NZ)
- **Upload files**: JSON API with `limit=20` (higher values cause empty responses)

## Steps
1. Install & import
2. Configuration
3. Utility functions
4. API client
5. Login
6. Query all data
7. Preview results
8. Export file list
9. Download all files
10. Logout

## Step 1: Install & Import

In [ ]:
!pip install requests pandas -q

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import hashlib
import hmac
import os
import time
import json
import re
from datetime import datetime, timedelta
from urllib.parse import urlparse, parse_qs, unquote
from IPython.display import display
import pandas as pd

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print("Ready.")

## Step 2: Configuration

In [ ]:
# ============================================================
# ACCOUNT CREDENTIALS
# ============================================================
ACCOUNT = "benl"
PASSWORD = "Realptt1"

# ============================================================
# QUERY TIME RANGE
# ============================================================
START_DATE = "2026-02-03"
END_DATE = "2026-02-18"

# ============================================================
# WHAT TO DOWNLOAD
# ============================================================
DOWNLOAD_VIDEO = True          # PTT video calls (MP4)
DOWNLOAD_AUDIO = True          # PTT radio recordings (WAV)
DOWNLOAD_UPLOAD_FILES = True   # Body camera files, pictures, etc.

# ============================================================
# DOWNLOAD OPTIONS
# ============================================================
OVERWRITE = True               # True = overwrite, False = skip existing
DOWNLOAD_DIR = "C:/Southbase Dropbox/Ben Lin/Ben's Folder/AI/Wearable Solution/dev_docs/aws_deployment/downloads"

# ============================================================
# TIMEZONE (DO NOT CHANGE UNLESS YOU MOVE COUNTRIES)
# ============================================================
TIMEZONE_OFFSET = -780         # NZDT: -780 minutes from GMT
TIME_DIFFERENCE_MS = 46800000  # 13 hours in ms (NZ server offset for audio)

# ============================================================
# TECHNICAL SETTINGS (DO NOT CHANGE)
# ============================================================
WEB_BASE = "https://realptt.com"   # Must use realptt.com, not api.realptt.com
UPLOAD_FILE_LIMIT = 20             # API max per page (higher = empty response)
CONNECT_TIMEOUT = 60
READ_TIMEOUT = 1800                # 30 min for large files
DOWNLOAD_RETRIES = 5
API_RETRIES = 3
API_RETRY_DELAY = 2

print("Configuration:")
print(f"  Account:    {ACCOUNT}")
print(f"  Date range: {START_DATE} to {END_DATE}")
print(f"  Download:   video={DOWNLOAD_VIDEO} audio={DOWNLOAD_AUDIO} uploads={DOWNLOAD_UPLOAD_FILES}")
print(f"  Overwrite:  {OVERWRITE}")
print(f"  Save to:    {DOWNLOAD_DIR}")

## Step 3: Utility Functions

In [ ]:
def safe_filename(name):
    """Remove invalid filename characters"""
    return re.sub(r'[<>:"/\\|?*]', '_', name)


def extract_file_url(down_path):
    """Extract real URL from uploadFile redirect URL"""
    try:
        parsed = urlparse(down_path)
        params = parse_qs(parsed.query)
        if 'FileUrl' in params:
            return unquote(params['FileUrl'][0])
        return down_path
    except:
        return down_path


def format_size(n):
    """Human-readable file size"""
    if n >= 1024**3: return f"{n/1024**3:.2f} GB"
    if n >= 1024**2: return f"{n/1024**2:.2f} MB"
    if n >= 1024:    return f"{n/1024:.2f} KB"
    return f"{n} B"


def format_speed(n):
    """Human-readable speed"""
    if n >= 1024**2: return f"{n/1024**2:.2f} MB/s"
    if n >= 1024:    return f"{n/1024:.2f} KB/s"
    return f"{n:.0f} B/s"


def format_eta(s):
    """Human-readable ETA"""
    if s < 60:   return f"{s:.0f}s"
    if s < 3600: return f"{s/60:.1f}min"
    return f"{s/3600:.1f}h"


def parse_js_args(arg_string):
    """Parse comma-separated JS function arguments, respecting single quotes"""
    args = []
    current = ''
    in_quote = False
    for ch in arg_string:
        if ch == "'" and not in_quote:
            in_quote = True
        elif ch == "'" and in_quote:
            in_quote = False
        elif ch == ',' and not in_quote:
            args.append(current.strip().strip("'"))
            current = ''
            continue
        current += ch
    args.append(current.strip().strip("'"))
    return args


def client_to_server_time(client_time_str):
    """Convert NZ client time to server time (subtract 13 hours)"""
    dt = datetime.strptime(client_time_str, '%Y-%m-%d %H:%M:%S')
    return dt - timedelta(milliseconds=TIME_DIFFERENCE_MS)


def date_no_pad(date_str):
    """Convert '2026-02-09' to '2026-2-9' (no zero-padding, for audio endpoint)"""
    dt = datetime.strptime(date_str, '%Y-%m-%d')
    return f"{dt.year}-{dt.month}-{dt.day}"


print("Utility functions ready.")

## Step 4: API Client

In [ ]:
class RealPTTClient:
    """
    REAL PTT Client v4 — uses realptt.com web endpoints.
    
    Three data sources:
    1. Video:  /ptt/webserver?event=org_videolist  (HTML, parse Ud() calls)
    2. Audio:  /ptt/webserver?event=org_audiolist  (HTML, per group/day)
    3. Upload: /ptt/uploadFile?method=get          (JSON, limit=20)
    """

    def __init__(self):
        self.session = requests.Session()
        retry = Retry(total=DOWNLOAD_RETRIES, backoff_factor=1,
                      status_forcelist=[429, 500, 502, 503, 504],
                      allowed_methods=["GET"])
        adapter = HTTPAdapter(max_retries=retry, pool_connections=10, pool_maxsize=20)
        self.session.mount("http://", adapter)
        self.session.mount("https://", adapter)
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        })
        self.session.verify = False
        self.groups = []

    # ------ Auth ------

    def login(self):
        """Login via realptt.com"""
        resp = self.session.get(f"{WEB_BASE}/ptt/random", timeout=30)
        random_str = resp.json()['data']['random']
        pwd_sha1 = hashlib.sha1(PASSWORD.encode()).hexdigest()
        pwd_enc = hmac.new(random_str.encode(), pwd_sha1.encode(), hashlib.sha1).hexdigest()
        resp = self.session.get(f"{WEB_BASE}/ptt/organization", params={
            'method': 'login', 'account': ACCOUNT,
            'pwd': pwd_enc, 'timeZoneOffset': TIMEZONE_OFFSET,
        }, timeout=30)
        assert resp.json().get('code') == 0, f"Login failed: {resp.json()}"
        print(f"Logged in as {ACCOUNT}")

    def logout(self):
        try:
            self.session.get(f"{WEB_BASE}/ptt/organization",
                             params={'method': 'logout'}, timeout=10)
        except: pass
        print("Logged out.")

    def fetch_groups(self):
        """Fetch all groups (needed for audio queries)"""
        resp = self.session.get(f"{WEB_BASE}/ptt/group", params={
            'method': 'get', 'limit': 20, 'page': 0,
        }, timeout=30)
        data = resp.json()
        self.groups = data.get('data', {}).get('groups', []) if data.get('code') == 0 else []
        print(f"Groups: {[(g['group_id'], g['group_name']) for g in self.groups]}")
        return self.groups

    # ------ 1. VIDEO ------

    def query_videos(self):
        """
        Query PTT video calls from org_videolist.
        Returns list of dicts: {id, time, src_account, src_name, type, url}
        """
        print("Querying videos...")
        resp = self.session.get(f"{WEB_BASE}/ptt/webserver", params={
            'event': 'org_videolist',
            'SearchItem': 'SearchAll', 'SearchAll': 1,
            'beginDate': START_DATE, 'endDate': END_DATE,
            'videoSaveTimeFlag': 1,
        }, timeout=60)
        if resp.status_code != 200:
            print(f"  HTTP {resp.status_code}")
            return []

        videos = []
        for match in re.findall(r'Ud\(([^)]+)\)', resp.text):
            args = parse_js_args(match)
            if len(args) >= 12 and args[11].startswith('http') and 'w3.org' not in args[11]:
                # Video times are SERVER time (UTC+8). Convert to NZ time (+13h).
                server_time = args[1]
                try:
                    server_dt = datetime.strptime(server_time, '%Y-%m-%d %H:%M:%S')
                    nz_dt = server_dt + timedelta(milliseconds=TIME_DIFFERENCE_MS)
                    nz_time = nz_dt.strftime('%Y-%m-%d %H:%M:%S')
                except:
                    nz_time = server_time
                videos.append({
                    'id': args[0], 'time': nz_time, 'server_time': server_time,
                    'src_account': args[4], 'src_name': args[6],
                    'type': args[9], 'url': args[11],
                })
        print(f"  Found {len(videos)} videos")
        return videos

    # ------ 2. AUDIO ------

    def query_audio(self):
        """
        Query PTT radio recordings from org_audiolist.
        Requires per-group, per-day queries.
        Returns list of dicts with download_url constructed.
        """
        print("Querying audio recordings...")
        if not self.groups:
            self.fetch_groups()

        # Build date list
        start = datetime.strptime(START_DATE, '%Y-%m-%d')
        end = datetime.strptime(END_DATE, '%Y-%m-%d')
        dates = []
        cur = start
        while cur <= end:
            dates.append(cur.strftime('%Y-%m-%d'))
            cur += timedelta(days=1)

        total_queries = len(dates) * len(self.groups)
        print(f"  {len(dates)} days x {len(self.groups)} groups = {total_queries} queries")

        all_audio = []
        n = 0
        for group in self.groups:
            gid = group['group_id']
            gname = group['group_name']
            for ds in dates:
                n += 1
                dfmt = date_no_pad(ds)
                resp = self.session.get(f"{WEB_BASE}/ptt/webserver", params={
                    'event': 'org_audiolist',
                    'GroupId': gid,
                    'time': f'{dfmt}_00:00:00',
                    'endTime': f'{dfmt}_23:59:59',
                    'pageSize': 20, 'sort': 0, 'autoPlay': 'false',
                }, timeout=60)

                if resp.status_code != 200 or len(resp.content) == 0:
                    continue

                for match in re.findall(r'Ud\((\d+[^)]+)\)', resp.text):
                    args = parse_js_args(match)
                    if len(args) >= 7:
                        spk_id = args[0]
                        client_time = args[1]
                        server_dt = client_to_server_time(client_time)
                        server_date = f"{server_dt.year}_{server_dt.month}_{server_dt.day}"
                        server_time_str = f"{server_dt.hour}_{server_dt.minute}_{server_dt.second}"

                        all_audio.append({
                            'spk_id': spk_id,
                            'time': client_time,
                            'user_name': args[2],
                            'group_name': args[3],
                            'duration': args[4],
                            'download_url': f"https://record.realptt.com/voice/?SpkId={spk_id}&time={server_date}&filename={server_date} {server_time_str}.wav&CodecType=0",
                            'query_group': gname,
                            'query_date': ds,
                        })

                count = len([1 for a in all_audio if a['query_date'] == ds and a['query_group'] == gname])
                if count > 0:
                    print(f"  [{n}/{total_queries}] {gname} / {ds}: {count} recordings")
                else:
                    print(f"  [{n}/{total_queries}] {gname} / {ds}...", end='\r')

                time.sleep(0.3)

        print(f"  Total audio: {len(all_audio)}                              ")
        return all_audio

    # ------ 3. UPLOAD FILES ------

    def query_upload_files(self):
        """
        Query uploaded files (body cam video, pictures, etc.) via JSON API.
        limit must be 20 — higher values cause empty responses.
        """
        print("Querying upload files...")
        all_files = []
        page = 1
        consecutive_fails = 0
        total_pages = None

        while True:
            params = {
                'method': 'get',
                'start_time': START_DATE,
                'end_time': END_DATE,
                'limit': UPLOAD_FILE_LIMIT,
                'page': page, 'sort': 0,
            }

            data = None
            for attempt in range(API_RETRIES + 1):
                try:
                    resp = self.session.get(f"{WEB_BASE}/ptt/uploadFile",
                                           params=params, timeout=(CONNECT_TIMEOUT, 60))
                    if len(resp.content) > 0:
                        data = resp.json()
                        break
                except: pass
                time.sleep(API_RETRY_DELAY * (attempt + 1))

            if data is None:
                consecutive_fails += 1
                if consecutive_fails >= 3:
                    break
                if total_pages and page < total_pages:
                    page += 1
                    continue
                break

            consecutive_fails = 0
            if data.get('code') != 0:
                break

            files = data.get('data', {}).get('uploadFiles', [])
            ps = data.get('data', {}).get('pageSize', 1)
            total_pages = ps

            if not files:
                break

            all_files.extend(files)
            print(f"  Page {page}/{ps}: {len(files)} items (Total: {len(all_files)})")

            if page >= ps:
                break
            page += 1
            time.sleep(0.5)

        # Categorize
        cats = {}
        for f in all_files:
            ft = f.get('file_type', 'Unknown')
            cats[ft] = cats.get(ft, 0) + 1
        print(f"  Total: {len(all_files)} — {cats}")

        return all_files


client = RealPTTClient()
print("Client ready (v4).")

## Step 5: Login

In [ ]:
client.login()
client.fetch_groups()

## Step 6: Query All Data

Audio queries require per-group per-day requests. For large date ranges this can take a minute.

In [ ]:
videos = []
audios = []
upload_files = []

if DOWNLOAD_VIDEO:
    videos = client.query_videos()

if DOWNLOAD_AUDIO:
    audios = client.query_audio()

if DOWNLOAD_UPLOAD_FILES:
    upload_files = client.query_upload_files()

print()
print("=" * 50)
print(f"Videos (PTT calls):      {len(videos)}")
print(f"Audio (radio recordings): {len(audios)}")
print(f"Upload files (body cam):  {len(upload_files)}")
print(f"TOTAL:                    {len(videos) + len(audios) + len(upload_files)}")
print("=" * 50)

## Step 7: Preview Results

In [ ]:
if videos:
    print(f"\nVideos ({len(videos)} items):")
    df = pd.DataFrame(videos)
    display(df[['time', 'src_account', 'src_name', 'type']].head(20))

if audios:
    print(f"\nAudio Recordings ({len(audios)} items):")
    df = pd.DataFrame(audios)
    display(df[['time', 'user_name', 'group_name', 'duration']].head(20))

if upload_files:
    print(f"\nUpload Files ({len(upload_files)} items):")
    df = pd.DataFrame(upload_files)
    cols = [c for c in ['upload_time', 'sender_account', 'file_type', 'file_name'] if c in df.columns]
    display(df[cols].head(20))

if not (videos or audios or upload_files):
    print("No data found. Check date range and re-run Step 6.")

## Step 8: Export File List

In [ ]:
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

export = {
    'export_time': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'config': {
        'start_date': START_DATE, 'end_date': END_DATE,
        'groups': [(g['group_id'], g['group_name']) for g in client.groups],
    },
    'summary': {
        'videos': len(videos), 'audio': len(audios),
        'upload_files': len(upload_files),
    },
    'videos': videos,
    'audio': audios,
    'upload_files': upload_files,
}

json_path = os.path.join(DOWNLOAD_DIR, 'file_list.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(export, f, ensure_ascii=False, indent=2)
print(f"File list exported: {json_path}")

## Step 9: Download All Files

In [ ]:
def download_file(session, url, save_path):
    """
    Download a single file with progress display and retry.
    Returns (success, message, file_size)
    """
    temp_path = save_path + '.downloading'

    if not OVERWRITE and os.path.exists(save_path):
        return True, "Exists (skipped)", os.path.getsize(save_path)

    for attempt in range(DOWNLOAD_RETRIES):
        try:
            if os.path.exists(temp_path):
                os.remove(temp_path)

            resp = session.get(url, timeout=(CONNECT_TIMEOUT, READ_TIMEOUT), stream=True)
            if resp.status_code not in (200, 206):
                if attempt < DOWNLOAD_RETRIES - 1:
                    time.sleep(2); continue
                return False, f"HTTP {resp.status_code}", 0

            total = int(resp.headers.get('content-length', 0))
            downloaded = 0
            t0 = time.time()
            last_print = t0

            with open(temp_path, 'wb') as f:
                for chunk in resp.iter_content(chunk_size=65536):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
                        now = time.time()
                        if now - last_print >= 0.5:
                            elapsed = now - t0
                            speed = downloaded / elapsed if elapsed > 0 else 0
                            if total > 0:
                                pct = downloaded / total * 100
                                eta = (total - downloaded) / speed if speed > 0 else 0
                                print(f"      {pct:.1f}% | {format_size(downloaded)}/{format_size(total)} | {format_speed(speed)} | ETA: {format_eta(eta)}   ", end='\r')
                            last_print = now

            print(" " * 80, end='\r')

            if downloaded == 0:
                if attempt < DOWNLOAD_RETRIES - 1:
                    time.sleep(2); continue
                return False, "Empty file", 0

            if os.path.exists(save_path):
                os.remove(save_path)
            os.rename(temp_path, save_path)

            elapsed = time.time() - t0
            speed = downloaded / elapsed if elapsed > 0 else 0
            return True, f"{format_size(downloaded)} @ {format_speed(speed)}", downloaded

        except requests.exceptions.Timeout:
            if attempt < DOWNLOAD_RETRIES - 1:
                print(f"\n      Timeout, retry {attempt+2}/{DOWNLOAD_RETRIES}")
                time.sleep(3); continue
            return False, "Timeout", 0
        except requests.exceptions.ConnectionError:
            if attempt < DOWNLOAD_RETRIES - 1:
                time.sleep(3); continue
            return False, "Connection error", 0
        except Exception as e:
            if attempt < DOWNLOAD_RETRIES - 1:
                time.sleep(2); continue
            return False, str(e)[:50], 0

    if os.path.exists(temp_path):
        try: os.remove(temp_path)
        except: pass
    return False, "Retry failed", 0


# ============================================================
# Download all files
# ============================================================
total_success = 0
total_failed = 0
total_bytes = 0
session = client.session

# --- 1. Videos ---
if videos:
    print(f"\n{'='*50}")
    print(f"DOWNLOADING VIDEOS ({len(videos)} items)")
    print(f"{'='*50}")
    save_dir = os.path.join(DOWNLOAD_DIR, 'video_ptt')
    os.makedirs(save_dir, exist_ok=True)

    for i, v in enumerate(videos, 1):
        ts = v['time'].replace(':', '-').replace(' ', '_')
        filename = safe_filename(f"{v['src_account']}_{ts}.mp4")
        save_path = os.path.join(save_dir, filename)

        print(f"[{i}/{len(videos)}] {filename}")
        ok, msg, sz = download_file(session, v['url'], save_path)
        if ok:
            total_success += 1; total_bytes += sz
            print(f"[{i}/{len(videos)}] OK: {msg}")
        else:
            total_failed += 1
            print(f"[{i}/{len(videos)}] FAILED: {msg}")

# --- 2. Audio ---
if audios:
    print(f"\n{'='*50}")
    print(f"DOWNLOADING AUDIO ({len(audios)} items)")
    print(f"{'='*50}")
    save_dir = os.path.join(DOWNLOAD_DIR, 'audio_ptt')
    os.makedirs(save_dir, exist_ok=True)

    for i, a in enumerate(audios, 1):
        ts = a['time'].replace(':', '-').replace(' ', '_')
        filename = safe_filename(f"{a['user_name']}_{a['group_name']}_{ts}.wav")
        save_path = os.path.join(save_dir, filename)

        print(f"[{i}/{len(audios)}] {filename}")
        ok, msg, sz = download_file(session, a['download_url'], save_path)
        if ok:
            total_success += 1; total_bytes += sz
            print(f"[{i}/{len(audios)}] OK: {msg}")
        else:
            total_failed += 1
            print(f"[{i}/{len(audios)}] FAILED: {msg}")

# --- 3. Upload Files ---
if upload_files:
    print(f"\n{'='*50}")
    print(f"DOWNLOADING UPLOAD FILES ({len(upload_files)} items)")
    print(f"{'='*50}")

    for i, fi in enumerate(upload_files, 1):
        dp = fi.get('down_path', '')
        if not dp:
            continue

        url = extract_file_url(dp)
        ft = fi.get('file_type', 'other').lower()

        if 'picture' in ft or 'image' in ft:
            sub, dext = 'upload_pictures', '.jpg'
        elif 'video' in ft:
            sub, dext = 'upload_video', '.mp4'
        elif 'audio' in ft:
            sub, dext = 'upload_audio', '.wav'
        else:
            sub, dext = 'upload_other', ''

        save_dir = os.path.join(DOWNLOAD_DIR, sub)
        os.makedirs(save_dir, exist_ok=True)

        sender = fi.get('sender_account', 'unknown')
        orig = fi.get('file_name', f'file_{i}')
        ext = os.path.splitext(urlparse(url).path)[1] or dext

        if orig.endswith(ext):
            filename = safe_filename(f"{sender}_{orig}")
        else:
            filename = safe_filename(f"{sender}_{orig}{ext}")

        save_path = os.path.join(save_dir, filename)

        print(f"[{i}/{len(upload_files)}] [{sub.split('_')[1]}] {filename[:55]}")
        ok, msg, sz = download_file(session, url, save_path)
        if ok:
            total_success += 1; total_bytes += sz
            print(f"[{i}/{len(upload_files)}] OK: {msg}")
        else:
            total_failed += 1
            print(f"[{i}/{len(upload_files)}] FAILED: {msg}")
        time.sleep(0.1)

# --- Summary ---
print(f"\n{'='*50}")
print("DOWNLOAD COMPLETE!")
print(f"  Success: {total_success}")
print(f"  Failed:  {total_failed}")
print(f"  Total:   {format_size(total_bytes)}")
print(f"  Saved:   {os.path.abspath(DOWNLOAD_DIR)}")
print(f"{'='*50}")

## Step 10: Logout

In [ ]:
client.logout()

print(f"\nOutput Structure:")
print(f"  {DOWNLOAD_DIR}/")
print(f"  +-- video_ptt/       # PTT video calls (MP4)")
print(f"  +-- audio_ptt/       # PTT radio recordings (WAV)")
print(f"  +-- upload_video/    # Body camera video (MP4)")
print(f"  +-- upload_audio/    # Uploaded audio")
print(f"  +-- upload_pictures/ # Pictures")
print(f"  +-- upload_other/    # Other uploads")
print(f"  +-- file_list.json   # Complete metadata")